# 🏛️ Rajasthan Government Circular RAG Pipeline
### End-to-End Retrieval-Augmented Generation for Devanagari Hindi Government Orders

This notebook reproduces the complete Multi-User RAG pipeline from `pdf_parser.py`, `vector_store.py`, `llm_client.py`, and `inspect_chunks.py` as sequential notebook cells. It ingests official government circulars (*"Mukhya Mantri Ayushman Jeevan Raksha Yojana"*), creates clause-aware vector embeddings, indexes them with local FAISS search, and generates grounded responses in formal Devanagari Hindi using Groq 70B LLM.

## Section 1: Setup & Environment Configuration
In this section, we load required dependencies (`PyMuPDF`, `sentence-transformers`, `faiss-cpu`, `groq`, `dotenv`) and verify that the `GROQ_API_KEY` is loaded.

In [ ]:
import os
import re
import json
import numpy as np
import fitz  # PyMuPDF
import faiss
from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer
from groq import Groq

# Load environment variables from .env file
load_dotenv()

groq_api_key = os.getenv('GROQ_API_KEY', '')
if groq_api_key and groq_api_key != 'your_groq_api_key_here':
    masked_key = groq_api_key[:4] + '...' + groq_api_key[-4:]
    print(f'[OK] GROQ_API_KEY loaded: {masked_key}')
else:
    print('[INFO] GROQ_API_KEY is not configured in .env file')

## Section 2: Clause-Aware PDF Parsing & Annexure Isolation
Government circulars contain structured numbered clauses (`1.`, `2.`, `3.`) and standalone Annexure sections (`ANNEXURE-I`, `ANNEXURE-II`). Our clause parser preserves complete numbered provisions and forces dedicated standalone chunks for Annexures.

In [ ]:
CLAUSE_PATTERN = re.compile(
    r'(?=(\n|^)(?:'
    r'[0-9]+[\.\)]|'
    r'[०-९]+[\.\)]|'
    r'\([0-9]+\)|'
    r'\([०-९]+\)|'
    r'ANNEXURE\s*[-–—]?\s*[I|V|X|0-9]+|'
    r'अनेक्सचर\s*[-–—]?\s*[I|V|X|0-9|०-९]+|'
    r'प्रकरण\s*[:\.]?|'
    r'शर्तें\s*[:\.]?|'
    r'पात्रता\s*[:\.]?|'
    r'उद्देश्य\s*[:\.]?'
    r'))',
    re.IGNORECASE
)

ANNEXURE_HEADER = re.compile(
    r'^(?:ANNEXURE\s*[-–—]?\s*[I|V|X|0-9]+|अनेक्सचर\s*[-–—]?\s*[I|V|X|0-9|०-९]+)',
    re.IGNORECASE
)

def clean_ocr_text(text: str) -> str:
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = re.sub(r'(\d+)\s+[\.]\s+', r'\1. ', text)
    text = re.sub(r'([०-९]+)\s+[\.]\s+', r'\1. ', text)
    return text.strip()

def parse_pdf_to_chunks(pdf_path: str, max_chunk_chars: int = 500) -> list:
    doc = fitz.open(pdf_path)
    chunks = []
    chunk_counter = 0

    for page_num in range(len(doc)):
        page_text = clean_ocr_text(doc[page_num].get_text('text'))
        splits = CLAUSE_PATTERN.split(page_text)
        clauses = [s.strip() for s in splits if s and s.strip()]

        current_chunk = ''
        clause_ref = None

        for clause in clauses:
            is_annexure = bool(ANNEXURE_HEADER.search(clause))
            match = re.match(r'^([0-9०-९]+[\.\)]|\([0-9०-९]+\)|ANNEXURE\s*[-–—]?\s*[I|V|X|0-9]+|अनेक्सचर\s*[-–—]?\s*[I|V|X|0-9]+)', clause, re.I)
            if match:
                clause_ref = match.group(1)

            if is_annexure:
                if current_chunk.strip():
                    chunks.append({
                        'id': f'chunk_{chunk_counter}',
                        'text': current_chunk.strip(),
                        'page': page_num + 1,
                        'clause': clause_ref or 'general'
                    })
                    chunk_counter += 1
                    current_chunk = ''
                chunks.append({
                    'id': f'chunk_{chunk_counter}',
                    'text': clause.strip(),
                    'page': page_num + 1,
                    'clause': clause_ref or 'annexure'
                })
                chunk_counter += 1
                continue

            if len(current_chunk) + len(clause) <= max_chunk_chars:
                current_chunk = (current_chunk + '\n\n' + clause).strip() if current_chunk else clause
            else:
                if current_chunk.strip():
                    chunks.append({
                        'id': f'chunk_{chunk_counter}',
                        'text': current_chunk.strip(),
                        'page': page_num + 1,
                        'clause': clause_ref or 'general'
                    })
                    chunk_counter += 1
                current_chunk = clause

        if current_chunk.strip():
            chunks.append({
                'id': f'chunk_{chunk_counter}',
                'text': current_chunk.strip(),
                'page': page_num + 1,
                'clause': clause_ref or 'general'
            })
            chunk_counter += 1

    return chunks

# Parse Rajasthan Finance Dept Government Order
pdf_path = './data/rajasthani/Mukhya_Mantri_Ayushman_Jeevan_Raksha_Yojana.pdf'
chunks = parse_pdf_to_chunks(pdf_path)

print(f'[PDF Parsed]: {pdf_path}')
print(f'[Extracted Chunks]: {len(chunks)}\n')
print('=' * 75)
for c in chunks:
    snippet = c['text'].replace('\n', ' ')
    if len(snippet) > 90:
        snippet = snippet[:90] + '...'
    print(f"[{c['id']}] Page {c['page']} | Clause: {c['clause']} | Length: {len(c['text'])} chars")
    print(f"      Snippet: {ascii(snippet)}")
    print('-' * 75)

## Section 3: Multilingual Embedding & FAISS Vector Indexing
Each text chunk is converted into a dense vector embedding using `sentence-transformers/paraphrase-multilingual-mpnet-base-v2`. Vectors are L2-normalized and indexed in FAISS (`IndexFlatIP`) for exact Cosine Similarity retrieval.

In [ ]:
embedding_model_name = 'sentence-transformers/paraphrase-multilingual-mpnet-base-v2'
print(f'[INFO] Loading embedding model: {embedding_model_name}...')
embedder = SentenceTransformer(embedding_model_name)

# Compute embeddings for all chunks
texts = [c['text'] for c in chunks]
embeddings = embedder.encode(texts, convert_to_numpy=True, show_progress_bar=False).astype('float32')

# L2 Normalize for Cosine Similarity using Inner Product
norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
norms[norms == 0] = 1e-10
normalized_embeddings = embeddings / norms

# Build FAISS IndexFlatIP index
dimension = embedder.get_sentence_embedding_dimension()
index = faiss.IndexFlatIP(dimension)
index.add(normalized_embeddings)

# Persist index and metadata locally under ./data/rajasthani/
os.makedirs('./data/rajasthani', exist_ok=True)
index_path = './data/rajasthani/faiss_index.bin'
metadata_path = './data/rajasthani/chunks.json'

faiss.write_index(index, index_path)
with open(metadata_path, 'w', encoding='utf-8') as f:
    json.dump(chunks, f, ensure_ascii=False, indent=2)

print('[OK] FAISS Index successfully built and saved!')
print(f'   - Index Path: {index_path}')
print(f'   - Vector Dimension: {dimension}')
print(f'   - Total Stored Vectors: {index.ntotal}')

## Section 4: Dense Semantic Vector Retrieval
The `retrieve(query, top_k)` function embeds user questions and performs inner-product similarity search against the FAISS index, retrieving top matching context chunks.

In [ ]:
def retrieve(query: str, top_k: int = 3) -> list:
    query_vec = embedder.encode([query], convert_to_numpy=True).astype('float32')
    q_norm = np.linalg.norm(query_vec, axis=1, keepdims=True)
    q_norm[q_norm == 0] = 1e-10
    query_vec = query_vec / q_norm

    scores, indices = index.search(query_vec, min(top_k, len(chunks)))

    results = []
    for score, idx in zip(scores[0], indices[0]):
        if idx != -1 and idx < len(chunks):
            item = dict(chunks[idx])
            item['score'] = float(score)
            results.append(item)
    return results

# Test Retrieval against sample Devanagari queries
test_queries = [
    'सड़क दुर्घटना में घायल व्यक्ति को कितनी राशि दी जाती है?',
    'अस्पताल द्वारा पोर्टल पर प्रविष्टि कितने समय में दर्ज करनी होती है?',
    'ANNEXURE-I के अंतर्गत अस्पताल के लिए क्या दिशा-निर्देश हैं?'
]

for q in test_queries:
    retrieved = retrieve(q, top_k=2)
    print(f'[Query]: {ascii(q)}')
    for r in retrieved:
        print(f"   [Score: {r['score']:.4f}] Chunk: {r['id']} | Clause: {r['clause']}")
    print('-' * 75)

## Section 5: Factual LLM Answer Generation (Groq 70B)
We pass retrieved context chunks to Groq's 70B model (`llama-3.3-70b-versatile`). The system prompt enforces **strict factual grounding** in formal Devanagari Hindi register, prohibiting outside knowledge and preserving exact figures (`रू0 10000/-`, `48 घंटे`).

In [ ]:
SYSTEM_PROMPT = """आप राजस्थान सरकार के आधिकारिक नियम/परिपत्र (जैसे "मुख्यमंत्री आयुष्मान जीवन रक्षा योजना") पर आधारित एक सटीक, तथ्यपरक सहायक हैं।

आपको नीचे केवल संबंधित संदर्भ (Context) दिया जा रहा है। उत्तर देते समय निम्न नियमों का सख्ती से पालन करें:

1. केवल प्रदान किए गए संदर्भ (Context) के आधार पर ही उत्तर दें। कोई बाहरी ज्ञान या अनुमान न लगाएं।
2. भाषा एवं शैली:
   - उत्तर का माध्यम और शैली स्रोत दस्तावेज के समान औपचारिक/सरकारी हिंदी (Devanagari script) होनी चाहिए।
   - यदि उपयोगकर्ता का प्रश्न अंग्रेजी में है, तब भी उत्तर को पूरी तरह हिंदी स्रोत पाठ पर ही आधारित रखें, परंतु उत्तर अंग्रेजी में दिया जा सकता है।
3. यदि प्रश्न का उत्तर दिए गए संदर्भ में उपलब्ध नहीं है, तो स्पष्ट रूप से लिखें कि "प्रदान किए गए संदर्भ में इस संबंध में जानकारी उपलब्ध नहीं है।"
4. सटीकता एवं विवरण:
   - उत्तर को संक्षिप्त और तथ्यपरक रखें।
   - विशिष्ट आंकड़ों, तिथियों, समय-सीमाओं (जैसे 48 घंटे, 10000/-, रू0 10000/-) और खंड/पैरा संख्याओं (Clause/Paragraph numbers) को स्रोत दस्तावेज के अनुसार ही सटीक उद्धृत करें।
"""

llm_client = Groq(api_key=groq_api_key) if groq_api_key and groq_api_key != 'your_groq_api_key_here' else None

def generate_answer(query: str, context_text: str) -> str:
    if not llm_client:
        return '[Error: GROQ_API_KEY is not configured in .env file]'
    
    user_msg = f'संदर्भ (Context):\n{context_text}\n\nप्रश्न (Query):\n{query}'
    
    response = llm_client.chat.completions.create(
        model='llama-3.3-70b-versatile',
        messages=[
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': user_msg}
        ],
        temperature=0.0,
        max_tokens=800
    )
    return response.choices[0].message.content.strip()

def rag_query(query_text: str, top_k: int = 3) -> dict:
    retrieved_chunks = retrieve(query_text, top_k=top_k)
    context_text = '\n\n'.join([c['text'] for c in retrieved_chunks])
    answer = generate_answer(query_text, context_text)
    return {
        'query': query_text,
        'answer': answer,
        'context': context_text
    }

## Section 6: Interactive End-to-End Demonstration
Run full RAG pipeline execution on a natural language query about the government order and inspect the retrieved context and generated answer side by side.

In [ ]:
# Demo Query
demo_query = 'सड़क दुर्घटना में घायल व्यक्ति को अस्पताल पहुँचाने पर कितनी प्रोत्साहन राशि दी जाती है?'

result = rag_query(demo_query, top_k=3)

print('=' * 80)
print(f"QUERY: {ascii(result['query'])}")
print('=' * 80)
print('\nGENERATED GROUNDED ANSWER (Formal Devanagari Hindi):')
print(ascii(result['answer']))
print('\n' + '-' * 80)
print('RETRIEVED CONTEXT (Raw Chunks Used for Auditing):')
print(ascii(result['context']))
print('=' * 80)